<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula05e%20-%20MLP%20Iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Aula 05e: Aplicação no Dataset Iris e Transição para o Scikit-Learn `MLPClassifier`

---

### 🎯 Testando no Mundo Real e Adotando as Bibliotecas Profissionais
Fechando o ciclo da construção artesanal:
1. Treinamos nossa rede neural profunda `MultiLayer([10])` no clássico dataset de flores **Iris** (Fisher, 1936).
2. Introduzimos o modelo oficial pronto para produção do Scikit-Learn: **`MLPClassifier`**!
3. Validamos com rigor científico usando **Validação Cruzada (5-Fold Cross-Validation)**!



## 1. Carregando o Dataset Iris (Flores: Setosa, Versicolor, Virginica)


In [259]:
import numpy as np
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)

## 2. Testando Nossa Rede `MultiLayer([10])` Feita do Zero no Dataset Iris

> Treinamento com 10 neurônios ocultos e função tangente hiperbólica.


In [271]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import accuracy_score

def include_bias(X):
  return np.c_[np.ones(X.shape[0]), X]

def one_hot_encode(y, labels):
  y_hot = np.ones((y.shape[0], len(labels)), dtype=int) * (-1)
  for i, label in enumerate(list(set(y))):
    idxs = np.where(y == label)[0]
    y_hot[idxs, i] = 1
  return y_hot

class MultiLayer(BaseEstimator, ClassifierMixin):
  def __init__(self, n_hidden=[2], max_iter=100000, learning_rate=0.001):
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.n_hidden = n_hidden
    self.activation = np.tanh

  def forward(self, X):
    self.A = []
    self.Z = []
    AUX = X.copy()
    for W in self.Ws:
      self.A.append(include_bias(AUX))
      self.Z.append(self.A[-1] @ W)
      AUX = self.activation(self.Z[-1])
    return AUX

  def backward(self, y, y_pred):
    grads = []
    output_delta = y_pred - y
    output_grad = self.A[-1].T @ output_delta
    grads.append(output_grad)
    for i in range(len(self.Ws)-1, 0, -1):
      tanh_grad = (1 - np.tanh(self.Z[i-1])**2)
      input_delta = output_delta @ self.Ws[i][1:].T * tanh_grad
      grad = self.A[i-1].T @ input_delta
      grads.insert(0, grad)
      output_delta = input_delta.copy()
    for i in range(len(self.Ws)):
      self.Ws[i] -= self.learning_rate * grads[i]
    return self

  def fit(self, X, y):
    self.Ws = []
    previous_output = X.shape[1]
    for n in self.n_hidden:
      self.Ws.append(np.random.randn(previous_output+1, n))
      previous_output = n
    self.labels = sorted(list(set(y)))
    y = one_hot_encode(y, self.labels)
    self.Ws.append(np.random.randn(previous_output+1, y.shape[1]))
    for _ in range(self.max_iter):
      logits = self.forward(X)
      self.backward(y, logits)
    return self

  def predict(self, X):
    logits = self.forward(X)
    idxs = np.argmax(logits, axis=1)
    return np.array([self.labels[idx] for idx in idxs])

model = MultiLayer([10])
model.fit(X, y)
y_pred = model.predict(X)
print(f"Accuracy: {accuracy_score(y, y_pred)}")

Accuracy: 0.9866666666666667


## 3. O Padrão da Indústria: `MLPClassifier` do Scikit-Learn

> O Scikit-Learn implementa o mesmo algoritmo com otimizadores avançados como o **Adam** e regularização $L_2$!


In [273]:
# O modelo oficial profissional de Redes Neurais do Scikit-Learn!
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(max_iter=1000, random_state=42)
model.fit(X, y)
y_pred = model.predict(X)
print(f"Acurácia do MLPClassifier do Scikit-Learn: {accuracy_score(y, y_pred) * 100:.2f}%")


Accuracy: 0.98


## 4. Avaliação Científica com Validação Cruzada (5 Folds)


In [275]:
from sklearn.model_selection import cross_validate

# Avaliação robusta e imparcial com Validação Cruzada de 5 Folds
scores = cross_validate(model, X, y, cv=5, scoring="accuracy")
print(f"Acurácia Média na Validação Cruzada (5 folds): {scores['test_score'].mean() * 100:.2f}%")
print("Desempenho em cada fold:", scores["test_score"].round(3))


0.9800000000000001
[1.         1.         0.96666667 0.93333333 1.        ]


---
## 🎓 Resumo da Aula 05e:
1. **Do Zero à Produção**: Vimos nossa implementação em NumPy atingir resultados comparáveis aos da biblioteca oficial `MLPClassifier`.
2. **Conclusão do Lote 4**: Dominamos completamente a arquitetura de Redes Multicamadas e o algoritmo de Backpropagation!

👉 **Próximo e Último Passo**: No **Lote 5 (Aula 06a)**, ingressaremos no ecossistema de Deep Learning profissional com **PyTorch**!

